# 01 — Hello World

Connect to a simulated robot, capture sensor data, and navigate to a goal.

**Prerequisites**: `pip install threewe[sim]`

**Backend**: This notebook uses the Gazebo simulation backend. No physical hardware needed.

In [ ]:
import asyncio
import numpy as np
from threewe import Robot

## Connect to the Robot

The `Robot` class is the single entry point. Set `backend="gazebo"` for simulation.
The async context manager handles connection and cleanup automatically.

In [ ]:
robot = Robot(backend="gazebo", scene="office_v2")
robot.connect()
print(f"Connected: {robot.is_connected}")
print(f"Backend: {robot.backend_name}")

## Read Sensors

All perception methods are synchronous and return numpy arrays or typed dataclasses.

In [ ]:
# Current pose in map frame
pose = robot.get_pose()
print(f"Pose: x={pose.x:.2f}, y={pose.y:.2f}, theta={pose.theta:.2f} rad")

# Camera image — (H, W, 3) uint8 numpy array, ready for VLM/CNN
image = robot.get_image()
print(f"Image shape: {image.shape}, dtype: {image.dtype}")

# LiDAR scan
scan = robot.get_lidar_scan()
print(f"LiDAR: {scan.ranges.shape[0]} points, range max={scan.range_max}m")

# IMU
imu = robot.get_imu()
print(f"IMU accel: {imu.acceleration}")

## Navigate to a Goal

`move_to()` uses Nav2 under the hood for path planning and execution.
It's async — in a notebook, use `await` or `asyncio.run()`.

In [ ]:
result = await robot.move_to(x=2.0, y=1.0)
print(f"Navigation {'succeeded' if result.success else 'failed'}")
print(f"Reason: {result.reason}")
print(f"Distance traveled: {result.distance:.2f}m in {result.duration:.1f}s")

# Check final pose
final_pose = robot.get_pose()
print(f"Final pose: x={final_pose.x:.2f}, y={final_pose.y:.2f}")

## Low-Level Velocity Control

For direct control (e.g., RL policy outputs), use `set_velocity()`.
The robot has mecanum wheels — it supports omnidirectional motion.

In [ ]:
# Drive forward at 0.2 m/s for 2 seconds
robot.set_velocity(vx=0.2, vy=0.0, omega=0.0)
await asyncio.sleep(2.0)
robot.stop()

print(f"Pose after forward drive: {robot.get_pose()}")

## Cleanup

In [ ]:
robot.disconnect()
print("Disconnected.")

## Next Steps

- **02_slam_exploration.ipynb** — Autonomous mapping
- **03_point_navigation.ipynb** — Multi-waypoint navigation
- **04_rl_training.ipynb** — Train a policy with Gymnasium
- **05_data_collection.ipynb** — Record trajectories for imitation learning